# Contador de itens em esteira — subtração de background

Avaliação I — Visão Computacional (prof. Lúcio Campos)

## Etapa 1 · parte 1 — leitura e inspeção dos vídeos

Esta parte cobre a leitura dos dois vídeos com OpenCV, os metadados e a inspeção de frames. Subtração de background (MOG2/KNN), limiarização, morfologia, contornos e contagem pela linha virtual entram nas próximas partes da Etapa 1; a avaliação é a Etapa 2.

**Colab:** envie `01.mp4` e `test_video.mp4` pelo painel de arquivos (vão para `/content`). O armazenamento do Colab some ao reiniciar o ambiente; para algo durável, monte o Drive e aponte `VIDEO_DIR` para a pasta com os vídeos.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("OpenCV", cv2.__version__)

try:
    import google.colab  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

# Colab: o painel de arquivos envia para /content (com o Drive montado: Path("/content/drive/MyDrive/..."))
VIDEO_DIR = Path("/content") if EM_COLAB else Path("../data/raw")
VIDEOS = {"01": VIDEO_DIR / "01.mp4", "test_video": VIDEO_DIR / "test_video.mp4"}

## 1. Funções de leitura

`cv2.VideoCapture` não levanta exceção quando o arquivo não abre: ele só devolve `isOpened() == False`. Por isso `abrir_video` checa explicitamente.

In [ ]:
def abrir_video(caminho):
    """Abre o vídeo ou falha com mensagem clara."""
    cap = cv2.VideoCapture(str(caminho))
    if not cap.isOpened():
        raise FileNotFoundError(f"Não foi possível abrir {caminho}")
    return cap


def metadados(caminho):
    """Resolução, fps e nº de frames declarado pelo container."""
    cap = abrir_video(caminho)
    meta = {
        "largura": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "altura": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        "fps": cap.get(cv2.CAP_PROP_FPS),
        "frames_declarados": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
    }
    cap.release()
    meta["duracao_s"] = meta["frames_declarados"] / meta["fps"] if meta["fps"] else float("nan")
    return meta


def contar_frames_lidos(caminho):
    """Decodifica o vídeo inteiro e conta os frames realmente lidos."""
    cap = abrir_video(caminho)
    n = 0
    while cap.read()[0]:
        n += 1
    cap.release()
    return n


def ler_frame(caminho, indice):
    """Frame `indice` do vídeo (BGR, uint8)."""
    cap = abrir_video(caminho)
    cap.set(cv2.CAP_PROP_POS_FRAMES, indice)
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise ValueError(f"Frame {indice} não pôde ser lido de {caminho}")
    return frame


def mostrar(frames, titulos):
    """Exibe frames BGR lado a lado (cv2.imshow não funciona no Colab)."""
    fig, eixos = plt.subplots(1, len(frames), figsize=(5 * len(frames), 4))
    for ax, f, t in zip(np.atleast_1d(eixos), frames, titulos):
        ax.imshow(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))  # OpenCV é BGR, matplotlib é RGB
        ax.set_title(t)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 2. Metadados

Compara o nº de frames **declarado** pelo container (`CAP_PROP_FRAME_COUNT`) com o nº de frames **realmente decodificados**.

In [ ]:
META = {}
for nome, caminho in VIDEOS.items():
    m = metadados(caminho)
    m["frames_lidos"] = contar_frames_lidos(caminho)
    META[nome] = m
    print(f"{nome:>10}: {m['largura']}x{m['altura']} | {m['fps']:.2f} fps | "
          f"declarados={m['frames_declarados']} lidos={m['frames_lidos']} | {m['duracao_s']:.1f} s")

## 3. Inspeção de frames

Três frames por vídeo (5%, 40% e 75% da duração) para ver a cena, a iluminação e o sentido do movimento.

In [ ]:
for nome, caminho in VIDEOS.items():
    n = META[nome]["frames_lidos"]
    indices = [int(n * f) for f in (0.05, 0.40, 0.75)]
    mostrar([ler_frame(caminho, i) for i in indices], [f"{nome} — frame {i}" for i in indices])

## 4. Observações da etapa 1

A câmera parece fixa nos dois vídeos; no `test_video.mp4`, porém, bordas de estruturas estáticas aparecem como foreground em alguns frames (seção 5), o que é compatível com uma leve instabilidade do fundo gerado. O `01.mp4` (352×288, 25 fps, 85,2 s) mostra, de cima, uma rampa de superfície corrugada por onde passam leitões brancos e escuros, em geral da esquerda para a direita. O `test_video.mp4` (1920×1080, 30 fps, 24,0 s) mostra maçãs numa esteira em perspectiva: os itens aumentam de tamanho do canto superior esquerdo ao inferior direito, sentido em que a esteira os leva.

Há elementos fixos sobre a imagem que o subtrator trataria como movimento. No `01.mp4`, o timestamp (canto superior esquerdo) muda a cada segundo e o rótulo "Camera 01" fica no canto inferior direito. No `test_video.mp4`, há uma marca "Veo" no canto inferior direito. Essas regiões são excluídas da máscara pelo parâmetro `ignorar`.

O `test_video.mp4` tem uma estrutura temporal peculiar. A diferença média absoluta entre os frames `i` e `i + 240` é de cerca de 1,3 nível de cinza, contra cerca de 5,5 entre `i` e `i + 239`, e há saltos bruscos nos frames 239→240 e 479→480. Isso indica um trecho de 8 s repetido três vezes. Além disso, um frame em cada cinco é repetido (144 de 719 pares consecutivos), o que sugere conversão de 24 para 30 fps; a taxa efetiva é de 24 fps. A marca "Veo" sugere que o vídeo foi gerado por IA, então a física das maçãs pode não ser fiel.

Nos leitões, a área de um animal isolado dentro da zona de contagem fica entre cerca de 3.700 e 6.400 px. Nas maçãs, o diâmetro é de 70 a 80 px na imagem reduzida a 50%, e a esteira nunca fica vazia, o que afeta o modelo de fundo (seção 8). A iluminação é discutida na seção 16 e a regra da contagem manual, na seção 13.

## Etapa 1 · parte 2 — subtração de background e máscara limpa

**Ideia.** Cada pixel tem um modelo do que é "fundo". O **MOG2** modela a intensidade de cada pixel como uma mistura de gaussianas e marca como foreground o pixel que não se encaixa em nenhuma componente de fundo. O **KNN** compara o pixel com amostras recentes e marca como foreground se há poucos vizinhos próximos. Nos dois, o modelo se adapta com o tempo.

`apply()` devolve uma máscara com 3 valores: **0** fundo, **127** sombra (com `detectShadows=True`) e **255** foreground. Limiarizar em ~200 descarta as sombras. Depois, **abertura** (erosão + dilatação) remove ruído pequeno e **fechamento** (dilatação + erosão) fecha buracos dentro dos objetos.

**Pré-condições e limitações.** Câmera fixa. O modelo precisa de alguns frames para estabilizar (*warm-up*). Objeto parado vira fundo depois de um tempo, e objetos encostados formam um único blob.

In [ ]:
# Parâmetros por vídeo. `ignorar`: regiões (x0, y0, x1, y1) em fração da imagem, p/ timestamp e marcas d'água.
PARAMS = {
    "01": dict(escala=1.0, blur=5, k_abrir=3, k_fechar=9,
               ignorar=[(0.0, 0.0, 0.70, 0.12), (0.70, 0.84, 1.0, 1.0)]),
    "test_video": dict(escala=0.5, blur=5, k_abrir=5, k_fechar=11,
                       ignorar=[(0.93, 0.92, 1.0, 1.0)]),
}
LIMIAR_FG = 200  # > 127 (sombra) e <= 255 (foreground)


def novo_subtrator(tipo, p):
    """MOG2 ou KNN com os parâmetros do vídeo.

    history e limiar de variância (MOG2) / distância (KNN); `fundo` = backgroundRatio do MOG2: fração do peso
    das gaussianas que conta como fundo (padrão 0.9; menor => só o modo dominante é fundo); `knn_k` = nº
    mínimo de amostras próximas para o KNN considerar o pixel fundo.
    """
    h = p.get("history", 500)
    cv2.setRNGSeed(0)  # o KNN sorteia amostras: fixar a semente torna a contagem reprodutível
    if tipo == "MOG2":
        sub = cv2.createBackgroundSubtractorMOG2(history=h, varThreshold=p.get("var", 25), detectShadows=True)
        sub.setBackgroundRatio(p.get("fundo", 0.9))
    else:
        sub = cv2.createBackgroundSubtractorKNN(history=h, dist2Threshold=p.get("dist2", 400.0), detectShadows=True)
        sub.setkNNSamples(p.get("knn_k", 2))
    return sub


def mascara_roi(forma, ignorar):
    """255 onde vale, 0 nas regiões a ignorar."""
    h, w = forma[:2]
    roi = np.full((h, w), 255, np.uint8)
    for x0, y0, x1, y1 in ignorar:
        roi[int(y0 * h):int(y1 * h), int(x0 * w):int(x1 * w)] = 0
    return roi


def limpar(bruta, k_abrir, k_fechar):
    """Limiariza (descarta sombra) e aplica abertura + fechamento."""
    binaria = cv2.threshold(bruta, LIMIAR_FG, 255, cv2.THRESH_BINARY)[1]
    abrir = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_abrir, k_abrir))
    fechar = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_fechar, k_fechar))
    binaria = cv2.morphologyEx(binaria, cv2.MORPH_OPEN, abrir)
    return cv2.morphologyEx(binaria, cv2.MORPH_CLOSE, fechar)


def percorrer(caminho, tipo, p, amostras=()):
    """Passa o vídeo inteiro pelo subtrator.

    Retorna a fração de foreground (máscara bruta == 255) por frame e
    {indice: (frame redimensionado, máscara bruta)} para os índices em `amostras`.
    """
    sub = novo_subtrator(tipo, p)
    cap = abrir_video(caminho)
    roi, razao, guardados, i = None, [], {}, 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if p["escala"] != 1.0:
            frame = cv2.resize(frame, None, fx=p["escala"], fy=p["escala"], interpolation=cv2.INTER_AREA)
        if roi is None:
            roi = mascara_roi(frame.shape, p["ignorar"])
        bruta = cv2.bitwise_and(sub.apply(cv2.GaussianBlur(frame, (p["blur"], p["blur"]), 0)), roi)
        razao.append(np.count_nonzero(bruta == 255) / np.count_nonzero(roi))
        if i in amostras:
            guardados[i] = (frame, bruta)
        i += 1
    cap.release()
    return np.array(razao), guardados


def mostrar_mascaras(nome, res, p):
    """Linhas = amostras; colunas = frame, MOG2 bruta, MOG2 limpa, KNN bruta, KNN limpa."""
    indices = sorted(res["MOG2"][1])
    fig, eixos = plt.subplots(len(indices), 5, figsize=(18, 3.3 * len(indices)), squeeze=False)
    for linha, i in zip(eixos, indices):
        frame = res["MOG2"][1][i][0]
        linha[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        linha[0].set_title(f"{nome} — frame {i}")
        for col, tipo in ((1, "MOG2"), (3, "KNN")):
            bruta = res[tipo][1][i][1]
            linha[col].imshow(bruta, cmap="gray", vmin=0, vmax=255)
            linha[col].set_title(f"{tipo} bruta (127 = sombra)")
            linha[col + 1].imshow(limpar(bruta, p["k_abrir"], p["k_fechar"]), cmap="gray", vmin=0, vmax=255)
            linha[col + 1].set_title(f"{tipo} limpa")
    for ax in eixos.ravel():
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 5. MOG2 vs KNN lado a lado

Roda os dois subtratores no vídeo inteiro e mostra a máscara bruta e a limpa em alguns frames.

In [ ]:
AMOSTRAS = {"01": [300, 851, 1500], "test_video": [120, 360, 600]}

RES = {}
for nome, caminho in VIDEOS.items():
    RES[nome] = {t: percorrer(caminho, t, PARAMS[nome], AMOSTRAS[nome]) for t in ("MOG2", "KNN")}
    mostrar_mascaras(nome, RES[nome], PARAMS[nome])

## 6. Fração de foreground ao longo do tempo

Serve para ver o *warm-up* do modelo, os momentos de atividade e variações bruscas (iluminação, ruído).

In [ ]:
AQUECIMENTO = 5  # no 1º frame o modelo está vazio e tudo vira foreground (estoura a escala do gráfico)

fig, eixos = plt.subplots(len(VIDEOS), 1, figsize=(12, 3.5 * len(VIDEOS)))
for ax, (nome, caminho) in zip(np.atleast_1d(eixos), VIDEOS.items()):
    t = np.arange(META[nome]["frames_lidos"]) / META[nome]["fps"]
    for tipo, (razao, _) in RES[nome].items():
        ax.plot(t[AQUECIMENTO:], 100 * razao[AQUECIMENTO:], label=tipo, lw=1)
    ax.set_title(nome)
    ax.set_xlabel("tempo (s)")
    ax.set_ylabel("foreground (% da ROI)")
    ax.legend()
plt.tight_layout()
plt.show()

## 7. Observações da parte 2

No `01.mp4`, MOG2 e KNN separaram o animal do fundo de forma parecida: um blob compacto sobre o leitão, com ruído residual nas estrias da rampa e áreas em cinza (classe sombra, valor 127) ao redor do animal, que a binarização descarta. Nas maçãs, com os parâmetros padrão, o KNN marcou mais foreground que o MOG2 (cerca de 20 a 35% da região útil, contra 10 a 18%), e os dois produziram máscaras com anéis e bordas em vez de maçãs preenchidas.

A abertura removeu pontos isolados (ruído das estrias e da compressão) e o fechamento preencheu buracos dentro do animal. Nas maçãs, o fechamento com kernel 11 uniu maçãs vizinhas em blobs em cadeia, e a abertura não os separou. É a primeira evidência de objetos encostados sendo tratados como um só.

No gráfico de foreground, o valor próximo de 100% no primeiro frame ocorre porque o modelo ainda está vazio; por isso os primeiros `AQUECIMENTO` frames são omitidos. No `01.mp4`, a atividade se concentra entre cerca de 30 s e 62 s, com picos de até 19% da região, e o resto do vídeo é quase vazio. No `test_video.mp4`, as quedas bruscas em 8,0 s e 16,0 s coincidem com os cortes entre as repetições do trecho de 8 s (seção 4).

Entre as falhas visíveis até aqui estão as bordas de estruturas estáticas do fundo marcadas como foreground nas maçãs, o ruído das estrias no `01.mp4` e as maçãs encostadas. Elas voltam na análise da Etapa 2 (seções 14 a 16).

## Etapa 1 · parte 3 — contornos, filtro de área e contagem por linha virtual

**Ideia.**
1. **Contornos:** `findContours` sobre a máscara limpa (só contornos externos). A **área** do contorno (`contourArea`, em pixels) descarta ruído: abaixo de `area_min` não é objeto.
2. **Centroide** do contorno pelos momentos de imagem: $c = (m_{10}/m_{00},\; m_{01}/m_{00})$.
3. **Linha virtual** por dois pontos $a$ e $b$. Com $n$ o vetor normal unitário apontando no sentido do fluxo, a **distância com sinal** de um ponto $p$ à linha é $s(p) = (p - a)\cdot n$. Um objeto **cruzou** quando o centroide passa de $s<0$ para $s\ge 0$ entre dois frames.
4. **Rastreio:** o mesmo objeto é reconhecido de um frame para o outro pelo centroide mais próximo (até `dmax` pixels). Cada trilha conta **uma única vez**. Quando dois objetos se fundem num contorno só, as duas trilhas passam a seguir esse contorno (senão a trilha que atravessa a linha se perderia).
5. **Zona de contagem:** a máscara só vale numa faixa em volta da linha. Isso corta ruído longe da linha e evita que uma fila de itens vire um contorno gigante.
6. **Objetos encostados:** se a área do contorno é bem maior que a de um item (`area_item`), o contorno conta `round(area / area_item)` itens, descontando os que já foram contados e continuam fundidos nele (no mínimo, uma contagem por trilha que cruzou).

**Pré-condições e limitações.** A contagem herda os defeitos da máscara: item partido em dois contornos conta dobrado, itens colados dependem da heurística de área, e o objeto que para vira fundo e some. A linha, a zona e os limiares são **por vídeo** (a imagem e o tamanho dos itens são bem diferentes).

In [ ]:
# Linha virtual e zona de contagem por vídeo (frações da imagem).
# a, b: extremos da linha | fluxo: direção do movimento (só o sentido importa) | margem: meia largura da zona (fração da largura)
# area_min / area_item em pixels da imagem já redimensionada | dmax: deslocamento máximo entre frames | aquecimento: frames ignorados
CONTAGEM = {
    "01": dict(a=(0.50, 0.12), b=(0.50, 0.88), fluxo=(1, 0), margem=0.20,
               area_min=400, area_item=5000, dmax=45, aquecimento=30),
    "test_video": dict(a=(0.646, 0.417), b=(0.5625, 0.704), fluxo=(1, 0.38), margem=0.075,
                       area_min=800, area_item=3500, dmax=30, aquecimento=30,
                       subtrator=dict(fundo=0.8, dist2=800.0)),  # esteira sempre cheia: ver a explicação na seção 8
}
AUSENTE_MAX = 5  # frames sem detecção antes de descartar a trilha


def geometria(forma, c):
    """Ponto da linha `a`, normal `n` (no sentido do fluxo) e polígono da zona, em pixels."""
    h, w = forma[:2]
    a = np.array([c["a"][0] * w, c["a"][1] * h])
    b = np.array([c["b"][0] * w, c["b"][1] * h])
    u = b - a
    n = np.array([-u[1], u[0]]) / np.linalg.norm(u)
    if n @ np.array(c["fluxo"], float) < 0:
        n = -n
    m = c["margem"] * w
    zona = np.array([a - m * n, a + m * n, b + m * n, b - m * n]).astype(np.int32)
    return a, n, zona


def detectar(binaria, area_min):
    """Contornos externos com área >= area_min -> [(cx, cy, area, contorno)]."""
    contornos, _ = cv2.findContours(binaria, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    dets = []
    for cnt in contornos:
        area = cv2.contourArea(cnt)
        if area >= area_min:
            m = cv2.moments(cnt)
            dets.append((m["m10"] / m["m00"], m["m01"] / m["m00"], area, cnt))
    return dets


def atualizar(trilhas, dets, a, n, c):
    """Cada trilha segue a detecção mais próxima (até `dmax`); várias trilhas podem seguir a mesma
    detecção quando os objetos se fundem. Devolve [(detecção, itens)] dos cruzamentos (- para +) deste frame."""
    for t in trilhas:
        t["ausente"] += 1
    seguidas = {}  # índice da detecção -> trilhas que a seguem
    for t in trilhas:
        if dets:
            dist = [np.hypot(d[0] - t["x"], d[1] - t["y"]) for d in dets]
            di = int(np.argmin(dist))
            if dist[di] <= c["dmax"]:
                seguidas.setdefault(di, []).append(t)
    cruzamentos = []
    for di, ts in seguidas.items():
        d = dets[di]
        lado = (np.array(d[:2]) - a) @ n
        novas = [t for t in ts if t["lado"] < 0 <= lado and not t["contada"]]
        ja_contadas = sum(t["contada"] for t in ts)
        for t in ts:
            t.update(x=d[0], y=d[1], lado=lado, ausente=0)
        if novas:
            for t in novas:
                t["contada"] = True
            # área de k itens vale k, descontando os que já foram contados e estão fundidos nesse contorno
            cruzamentos.append((d, max(len(novas), round(d[2] / c["area_item"]) - ja_contadas)))
    for di, d in enumerate(dets):
        if di not in seguidas:
            lado = (np.array(d[:2]) - a) @ n
            trilhas.append(dict(x=d[0], y=d[1], lado=lado, ausente=0, contada=lado >= 0))  # já nasceu do outro lado
    trilhas[:] = [t for t in trilhas if t["ausente"] <= AUSENTE_MAX]
    return cruzamentos


def desenhar(frame, a, n, zona, dets, total):
    """Frame com zona (azul), linha (vermelha), contornos (verde), centroides e contador."""
    img = frame.copy()
    cv2.polylines(img, [zona], True, (255, 128, 0), 1)
    p1, p2 = (zona[0] + zona[1]) // 2, (zona[3] + zona[2]) // 2
    cv2.line(img, tuple(int(v) for v in p1), tuple(int(v) for v in p2), (0, 0, 255), 2)
    for cx, cy, _, cnt in dets:
        cv2.drawContours(img, [cnt], -1, (0, 255, 0), 1)
        cv2.circle(img, (int(cx), int(cy)), 3, (0, 255, 255), -1)
    cv2.putText(img, f"contagem: {total}", (8, img.shape[0] - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    return img


def contar_objetos(caminho, tipo, p, c, anotar=(), mudanca_luz=None):
    """Percorre o vídeo e conta os objetos que cruzam a linha.

    Os parâmetros de `c["subtrator"]` (se houver) sobrescrevem os de `p`.
    `mudanca_luz=(frame, fator)`: multiplica o brilho dos frames a partir de `frame` (teste de iluminação).
    Retorna dict com: total, eventos [{frame, x, y, area, itens, recorte}],
    curva (contagem acumulada por frame) e anotados {indice: frame desenhado}.
    """
    p = {**p, **c.get("subtrator", {})}
    sub = novo_subtrator(tipo, p)
    cap = abrir_video(caminho)
    roi = a = n = zona = None
    trilhas, eventos, curva, anotados, total, i = [], [], [], {}, 0, 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if p["escala"] != 1.0:
            frame = cv2.resize(frame, None, fx=p["escala"], fy=p["escala"], interpolation=cv2.INTER_AREA)
        if mudanca_luz and i >= mudanca_luz[0]:
            frame = cv2.convertScaleAbs(frame, alpha=mudanca_luz[1])
        if roi is None:
            a, n, zona = geometria(frame.shape, c)
            faixa = np.zeros(frame.shape[:2], np.uint8)
            cv2.fillPoly(faixa, [zona], 255)
            roi = cv2.bitwise_and(mascara_roi(frame.shape, p["ignorar"]), faixa)
        suave = cv2.GaussianBlur(frame, (p["blur"], p["blur"]), 0)
        bruta = cv2.bitwise_and(sub.apply(suave), roi)
        dets = detectar(limpar(bruta, p["k_abrir"], p["k_fechar"]), c["area_min"]) if i >= c["aquecimento"] else []
        for (cx, cy, area, _), itens in atualizar(trilhas, dets, a, n, c):
            total += itens
            r = int(1.3 * np.sqrt(c["area_item"]))
            y0, x0 = max(int(cy) - r, 0), max(int(cx) - r, 0)
            recorte = cv2.resize(frame[y0:int(cy) + r, x0:int(cx) + r], (64, 64))
            eventos.append(dict(frame=i, x=cx, y=cy, area=area, itens=itens, recorte=recorte))
        curva.append(total)
        if i in anotar:
            anotados[i] = desenhar(frame, a, n, zona, dets, total)
        i += 1
    cap.release()
    return dict(total=total, eventos=eventos, curva=np.array(curva), anotados=anotados)


def galeria(eventos, colunas=12):
    """Um recorte por objeto contado (rótulo = frame), para conferir a contagem a olho."""
    linhas = -(-len(eventos) // colunas)
    fig, eixos = plt.subplots(linhas, colunas, figsize=(1.4 * colunas, 1.6 * linhas), squeeze=False)
    for ax in eixos.ravel():
        ax.axis("off")
    for ax, e in zip(eixos.ravel(), eventos):
        ax.imshow(cv2.cvtColor(e["recorte"], cv2.COLOR_BGR2RGB))
        ax.set_title(f"f{e['frame']}" + (f" ×{e['itens']}" if e["itens"] > 1 else ""), fontsize=7)
    plt.tight_layout()
    plt.show()

## 8. Contagem nos dois vídeos (MOG2 e KNN)

Roda a contagem com os dois subtratores e mostra: o total, os frames anotados (zona azul, linha vermelha, contornos verdes) e a contagem acumulada no tempo.

**Ajuste do subtrator no vídeo das maçãs.** A esteira nunca fica vazia, então o modelo de fundo aprende as próprias maçãs. Com o `backgroundRatio` padrão (0,9), as cores das maçãs entram no modelo de fundo e a máscara fica só com anéis e bordas. Baixar `fundo` no MOG2 deixa só o modo dominante (a esteira) como fundo, de modo que o interior das maçãs passa a ser foreground. No KNN, `dist2` age no sentido oposto: aceita como fundo pixels mais distantes das amostras, e o valor foi ajustado de 400 para 800 por tentativa e erro. Por isso `CONTAGEM["test_video"]` sobrescreve esses dois parâmetros, e a seção 11 mostra como o total reage a eles. As máscaras da seção 5 usam os parâmetros **padrão**, não os da contagem.

**Viés da calibração.** A linha, a zona, `area_min`, `area_item`, `dmax` e esse ajuste do subtrator foram escolhidos olhando estes mesmos dois vídeos (inclusive com apoio da imagem espaço-tempo da seção 12). Não há vídeo separado para validar, então o erro medido na Etapa 2 é **otimista**.

In [ ]:
ANOTAR = {"01": [925, 1139, 1459], "test_video": [150, 400, 650]}

CONT = {}
for nome, caminho in VIDEOS.items():
    CONT[nome] = {t: contar_objetos(caminho, t, PARAMS[nome], CONTAGEM[nome], ANOTAR[nome]) for t in ("MOG2", "KNN")}
    print(f"{nome:>10}: " + " | ".join(f"{t} = {r['total']}" for t, r in CONT[nome].items()))

for nome in VIDEOS:
    imgs = CONT[nome]["MOG2"]["anotados"]
    mostrar([imgs[i] for i in sorted(imgs)], [f"{nome} — MOG2, frame {i}" for i in sorted(imgs)])

In [ ]:
fig, eixos = plt.subplots(len(VIDEOS), 1, figsize=(12, 3.5 * len(VIDEOS)))
for ax, nome in zip(np.atleast_1d(eixos), VIDEOS):
    t = np.arange(len(CONT[nome]["MOG2"]["curva"])) / META[nome]["fps"]
    for tipo, r in CONT[nome].items():
        ax.step(t, r["curva"], where="post", label=f"{tipo} (total {r['total']})")
    ax.set_title(nome)
    ax.set_xlabel("tempo (s)")
    ax.set_ylabel("contagem acumulada")
    ax.legend()
plt.tight_layout()
plt.show()

## 9. Conferência visual: um recorte por objeto contado

Cada miniatura é um objeto contado pelo MOG2 (rótulo = frame; `×k` = contorno grande contado como k itens). Serve para achar contagens duplas, itens perdidos e blobs mal divididos.

In [ ]:
for nome in VIDEOS:
    print(nome)
    galeria(CONT[nome]["MOG2"]["eventos"])

## 10. Consistência no vídeo das maçãs

O `test_video.mp4` é um trecho de 8 s repetido 3 vezes (frames 0–239, 240–479, 480–719): os mesmos itens passam pela linha nas três repetições. Se a contagem fosse perfeita, o número seria igual nas três. A diferença entre os trechos mede a **instabilidade** da contagem (ruído da máscara, aquecimento do modelo, fusões).

In [ ]:
TRECHO = 240
for tipo, r in CONT["test_video"].items():
    por_trecho = np.zeros(3, int)
    for e in r["eventos"]:
        por_trecho[min(e["frame"] // TRECHO, 2)] += e["itens"]
    print(f"{tipo}: por trecho de 8 s = {por_trecho.tolist()} | dispersão (máx - mín) = {np.ptp(por_trecho)} | total {r['total']}")

## 11. Sensibilidade do total aos parâmetros do subtrator (maçãs)

Mesma contagem, variando só o parâmetro que mais pesa em cada subtrator: `fundo` (`backgroundRatio`) no MOG2 e `dist2` no KNN. Mostra o quanto o resultado depende da calibração.

In [ ]:
SENSIBILIDADE = [("MOG2", "fundo", f) for f in (0.9, 0.8, 0.7, 0.5)] + [("KNN", "dist2", d) for d in (400.0, 800.0, 1500.0)]

print("test_video — total contado em função do parâmetro do subtrator:")
for tipo, chave, valor in SENSIBILIDADE:
    c = {**CONTAGEM["test_video"], "subtrator": {chave: valor}}
    r = contar_objetos(VIDEOS["test_video"], tipo, PARAMS["test_video"], c)
    usado = "  <- usado na contagem" if CONTAGEM["test_video"]["subtrator"].get(chave) == valor else ""
    print(f"  {tipo} {chave} = {valor}: total = {r['total']}{usado}")

### Observações da parte 3

Uma leitura visual preliminar da imagem espaço-tempo (seção 12) indica cerca de 12 passagens no `01.mp4` e cerca de 17 por trecho de 8 s no `test_video.mp4` (cerca de 51 nos 24 s). Essa leitura não substitui a contagem manual da seção 13. Em comparação com ela, o MOG2 contou 10 e o KNN 11 nos leitões, e 59 (MOG2) e 43 (KNN) nas maçãs.

Nos leitões, a conferência por recortes mostra um contorno com dois animais encostados contado como um só (frame 923), o que explica parte da subcontagem. Nas maçãs, o resultado depende da calibração do subtrator. Variando só `fundo` no MOG2, o total foi 36 (0,9), 59 (0,8), 48 (0,7) e 29 (0,5); no KNN, `dist2` = 400, 800 e 1500 deu 10, 43 e 22. Não há faixa larga estável, e os valores usados na contagem foram escolhidos olhando os mesmos vídeos.

As três repetições do trecho de 8 s deveriam dar o mesmo número, porque os mesmos itens passam pela linha. O MOG2 contou [21, 21, 17] e o KNN [12, 17, 14], uma dispersão de 4 e 5 itens. As causas prováveis, ainda não isoladas, são a evolução do modelo de fundo entre as repetições, a formação do modelo no início do primeiro trecho e a variação, com o estado do modelo, do número de contornos fundidos ou parciais.

## 12. Imagem espaço-tempo da linha (apoio à contagem manual)

Cada coluna é um frame e cada linha é um ponto ao longo da linha virtual. Um objeto que atravessa a linha aparece como uma **mancha**. Esta imagem não depende do subtrator, então serve para conferir a contagem sem usar o algoritmo: conte as manchas de cada vídeo. No vídeo das maçãs há dois "corredores" lado a lado, e o trecho de 8 s se repete 3 vezes.

In [ ]:
def espaco_tempo(caminho, p, c, pontos=300):
    """Imagem (pontos da linha x frames) com os pixels amostrados ao longo da linha virtual."""
    cap = abrir_video(caminho)
    colunas, xs = [], None
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if p["escala"] != 1.0:
            frame = cv2.resize(frame, None, fx=p["escala"], fy=p["escala"], interpolation=cv2.INTER_AREA)
        if xs is None:
            h, w = frame.shape[:2]
            xs = np.linspace(c["a"][0] * w, c["b"][0] * w, pontos).astype(int)
            ys = np.linspace(c["a"][1] * h, c["b"][1] * h, pontos).astype(int)
        colunas.append(frame[ys, xs])
    cap.release()
    return np.stack(colunas, axis=1)


fig, eixos = plt.subplots(len(VIDEOS), 1, figsize=(14, 3.2 * len(VIDEOS)))
for ax, (nome, caminho) in zip(np.atleast_1d(eixos), VIDEOS.items()):
    img = espaco_tempo(caminho, PARAMS[nome], CONTAGEM[nome])
    duracao = img.shape[1] / META[nome]["fps"]
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), aspect="auto", extent=[0, duracao, img.shape[0], 0])
    ax.set_title(f"{nome} — linha virtual ao longo do tempo")
    ax.set_xlabel("tempo (s)")
    ax.set_ylabel("posição na linha (a → b)")
plt.tight_layout()
plt.show()

## 13. Etapa 2 — avaliação contra a contagem manual

**Esta é a única parte que depende de você:** conte os objetos à mão em cada vídeo e preencha `CONTAGEM_MANUAL` na célula abaixo. A imagem espaço-tempo da seção 12 ajuda a contar sempre do mesmo jeito.

Regra de contagem usada pelo algoritmo, e sugerida para a contagem manual: um objeto é contado **uma vez**, quando o seu centro cruza a linha virtual no sentido do fluxo (esquerda para direita nos leitões; do canto superior esquerdo para o inferior direito nas maçãs). Se usar outra regra, descreva-a aqui. No `test_video.mp4`, se contou apenas um trecho de 8 s, informe o intervalo de frames em `INTERVALO_MANUAL` (por exemplo, `(0, 240)`), e a comparação usará só os eventos desse intervalo.

Erro absoluto: $|auto - manual|$. Erro percentual: $|auto - manual| / manual \times 100$.

In [ ]:
# >>> PREENCHER: contagem manual de cada vídeo (número inteiro) e o intervalo de frames em que foi feita.
CONTAGEM_MANUAL = {"01": None, "test_video": None}
INTERVALO_MANUAL = {"01": (0, None), "test_video": (0, None)}  # (frame inicial, frame final); None = até o fim


def total_no_intervalo(r, ini, fim):
    """Itens contados pelo algoritmo entre os frames [ini, fim)."""
    return sum(e["itens"] for e in r["eventos"] if e["frame"] >= ini and (fim is None or e["frame"] < fim))


print(f"{'vídeo':>10} {'subtrator':>9} {'auto':>5} {'manual':>6} {'auto-manual':>11} {'erro abs.':>9} {'erro %':>7}")
for nome in VIDEOS:
    for tipo, r in CONT[nome].items():
        auto = total_no_intervalo(r, *INTERVALO_MANUAL[nome])
        manual = CONTAGEM_MANUAL[nome]
        if manual is None:
            print(f"{nome:>10} {tipo:>9} {auto:>5}   (falta preencher CONTAGEM_MANUAL)")
        else:
            print(f"{nome:>10} {tipo:>9} {auto:>5} {manual:>6} {auto - manual:>+11d} {abs(auto - manual):>9d} {100 * abs(auto - manual) / manual:>6.1f}%")

## 14. Análise de falhas: sombras

Mede, dentro da zona de contagem, a fração de pixels classificados como **foreground** (255) e como **sombra** (127) pelo subtrator, frame a frame.

In [ ]:
# Evidências de falhas: perfil por frame (foreground, sombra, brilho) com os mesmos parâmetros da contagem
def perfil_quadros(caminho, tipo, p, c, mudanca_luz=None):
    """Por frame, dentro da zona de contagem: % de foreground (255), % de sombra (127) e brilho médio (cinza).

    `mudanca_luz=(frame, fator)` multiplica o brilho dos frames a partir de `frame` (teste de iluminação).
    """
    p = {**p, **c.get("subtrator", {})}
    sub = novo_subtrator(tipo, p)
    cap = abrir_video(caminho)
    roi, fg, sombra, brilho, i = None, [], [], [], 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if p["escala"] != 1.0:
            frame = cv2.resize(frame, None, fx=p["escala"], fy=p["escala"], interpolation=cv2.INTER_AREA)
        if mudanca_luz and i >= mudanca_luz[0]:
            frame = cv2.convertScaleAbs(frame, alpha=mudanca_luz[1])
        if roi is None:
            faixa = np.zeros(frame.shape[:2], np.uint8)
            cv2.fillPoly(faixa, [geometria(frame.shape, c)[2]], 255)
            roi = cv2.bitwise_and(mascara_roi(frame.shape, p["ignorar"]), faixa)
            area_roi = np.count_nonzero(roi)
        bruta = cv2.bitwise_and(sub.apply(cv2.GaussianBlur(frame, (p["blur"], p["blur"]), 0)), roi)
        fg.append(100 * np.count_nonzero(bruta == 255) / area_roi)
        sombra.append(100 * np.count_nonzero(bruta == 127) / area_roi)
        brilho.append(cv2.mean(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), mask=roi)[0])
        i += 1
    cap.release()
    return dict(fg=np.array(fg), sombra=np.array(sombra), brilho=np.array(brilho))


PERFIL = {nome: {t: perfil_quadros(VIDEOS[nome], t, PARAMS[nome], CONTAGEM[nome]) for t in ("MOG2", "KNN")} for nome in VIDEOS}

print(f"{'vídeo':>10} {'subtrator':>9} {'foreground (% da zona)':>23} {'sombra (% da zona)':>19} {'sombra / (sombra + fg)':>23}")
for nome in VIDEOS:
    for tipo, pf in PERFIL[nome].items():
        a = CONTAGEM[nome]["aquecimento"]
        fg, sb = pf["fg"][a:].mean(), pf["sombra"][a:].mean()
        print(f"{nome:>10} {tipo:>9} {fg:>22.2f}% {sb:>18.2f}% {100 * sb / (sb + fg):>22.1f}%")

fig, eixos = plt.subplots(len(VIDEOS), 2, figsize=(14, 3.2 * len(VIDEOS)))
for linha, nome in zip(eixos, VIDEOS):
    t = np.arange(len(PERFIL[nome]["MOG2"]["fg"])) / META[nome]["fps"]
    for tipo, pf in PERFIL[nome].items():
        linha[0].plot(t, pf["fg"], label=tipo, lw=1)
        linha[1].plot(t, pf["sombra"], label=tipo, lw=1)
    linha[0].set_title(f"{nome} — foreground (255)")
    linha[1].set_title(f"{nome} — sombra (127)")
    for ax in linha:
        ax.set_xlabel("tempo (s)")
        ax.set_ylabel("% da zona de contagem")
        ax.legend()
plt.tight_layout()
plt.show()

### Análise: sombras

No `01.mp4`, 64% dos pixels marcados como foreground ou sombra caem na classe sombra (4,6% da zona contra 2,5% de foreground, no MOG2), e o KNN dá resultado quase igual. A maior parte dessa sombra não parece ser sombra de objeto. Os picos de até 100% da zona em cerca de 12 s, 18–19 s e 76–78 s coincidem com quedas bruscas do brilho médio (seção 16), e durante a passagem dos leitões (35 s a 50 s) a sombra chega a 60–75% da zona, acima do foreground (35–42%). Isso é compatível com a correção automática de exposição da câmera: quando o brilho global muda, o chão passa a ser classificado como sombra. Como a binarização em 200 descarta a classe 127, parte do corpo de leitões escuros também pode ser descartada; essa hipótese não foi testada.

Nas maçãs, a sombra ocupa 11% da zona e 20 a 22% do total de foreground mais sombra, oscilando entre cerca de 5% e 18% sem picos isolados. Os cortes em 8 s e 16 s aparecem como quedas nos dois painéis.

Para a contagem, os dois erros existem e têm sentidos opostos: sombra classificada como foreground (255) incha o contorno e a área, e sombra classificada como 127 é descartada.

## 15. Análise de falhas: objetos encostados e contornos parciais

Dois indicadores sobre os eventos de contagem: **contornos fundidos** (a área equivale a mais de um item, contados por `round(área / area_item)`) e **contornos parciais** (área abaixo de 60% de `area_item`: o subtrator preencheu só parte do objeto, ou um objeto foi partido em mais de um contorno).

In [ ]:
for nome in VIDEOS:
    ai = CONTAGEM[nome]["area_item"]
    for tipo, r in CONT[nome].items():
        ev = r["eventos"]
        fundidos = [e for e in ev if e["itens"] > 1]
        parciais = [e for e in ev if e["area"] < 0.6 * ai]
        de_fundidos = sum(e["itens"] for e in fundidos)
        print(f"{nome:>10} {tipo:>5}: total {r['total']:>2} em {len(ev):>2} eventos | "
              f"{de_fundidos:>2} itens ({100 * de_fundidos / r['total']:.0f}% do total) vieram de {len(fundidos)} contornos fundidos | "
              f"{len(parciais)} eventos parciais ({100 * len(parciais) / len(ev):.0f}%)")

for nome in VIDEOS:
    fundidos = [e for e in CONT[nome]["MOG2"]["eventos"] if e["itens"] > 1]
    print(f"\n{nome} — contornos fundidos contados pelo MOG2 (×k = k itens):")
    if fundidos:
        galeria(fundidos)

### Análise: objetos encostados e contornos parciais

Nos leitões, só um contorno é fundido segundo esse indicador (3 itens, 30% do total no MOG2, um grupo de três animais no frame 1139) e nenhum evento é parcial. O indicador não captura o frame 923, cujo recorte mostra dois animais num contorno contado como um item, porque a regra desconta um item já contado que estava fundido nesse contorno. Nas maçãs, 47 dos 59 itens do MOG2 (80%) vêm de 19 contornos fundidos, e 10 dos 31 eventos (32%) têm área abaixo de 60% de uma maçã. No KNN, 24 dos 43 itens (56%) vêm de 10 contornos fundidos, e 15 dos 29 eventos (52%) são parciais.

Portanto, o total das maçãs é em boa parte a área do contorno dividida pela área de um item, e não uma contagem de objetos separados. A heurística acerta quando a área de uma maçã é regular e a máscara a cobre por inteiro. Como a máscara cobre só parte de cada maçã (seção 5), a mesma maçã pode contar menos ou mais conforme o preenchimento naquele frame. Os recortes mostram contornos com duas ou três maçãs encostadas, em duas fileiras lado a lado, como a imagem espaço-tempo da seção 12 já indicava.

## 16. Análise de falhas: variação de iluminação

Primeiro, o brilho médio da zona ao longo do tempo (variação natural). Depois, um **teste controlado**: o brilho dos frames é multiplicado por `GANHO` a partir da metade do vídeo (mudança brusca de iluminação) e a contagem do MOG2 é refeita.

In [ ]:
GANHO = 1.3

fig, eixos = plt.subplots(len(VIDEOS), 2, figsize=(14, 3.2 * len(VIDEOS)))
for linha, (nome, caminho) in zip(eixos, VIDEOS.items()):
    n = META[nome]["frames_lidos"]
    mudanca = (n // 2, GANHO)
    base = PERFIL[nome]["MOG2"]
    luz = perfil_quadros(caminho, "MOG2", PARAMS[nome], CONTAGEM[nome], mudanca_luz=mudanca)
    total_luz = contar_objetos(caminho, "MOG2", PARAMS[nome], CONTAGEM[nome], mudanca_luz=mudanca)["total"]
    t = np.arange(n) / META[nome]["fps"]
    print(f"{nome:>10}: brilho original min {base['brilho'].min():.0f} / média {base['brilho'].mean():.0f} / máx {base['brilho'].max():.0f} | "
          f"total sem mudança = {CONT[nome]['MOG2']['total']}, com ganho x{GANHO} a partir de {t[mudanca[0]]:.1f} s = {total_luz}")
    linha[0].plot(t, base["brilho"], label="original", lw=1)
    linha[0].plot(t, luz["brilho"], label=f"ganho x{GANHO}", lw=1)
    linha[1].plot(t, base["fg"], label="original", lw=1)
    linha[1].plot(t, luz["fg"], label=f"ganho x{GANHO}", lw=1)
    linha[0].set_title(f"{nome} — brilho médio da zona")
    linha[1].set_title(f"{nome} — foreground do MOG2 (% da zona)")
    for ax in linha:
        ax.set_xlabel("tempo (s)")
        ax.legend()
plt.tight_layout()
plt.show()

### Análise: iluminação

A iluminação natural é estável nas maçãs (brilho médio da zona entre 93 e 121, oscilando com a passagem dos frutos). No `01.mp4`, o brilho fica perto de 160 até cerca de 33 s, com quedas curtas em 12 s e 18–19 s, cai para 119–145 durante a passagem dos leitões e se estabiliza em cerca de 150 depois de 50 s, com mais uma queda curta em 76–77 s.

O teste controlado (ganho de 1,3 na segunda metade do vídeo) teve efeitos diferentes. No `01.mp4`, o foreground do MOG2 salta para cerca de 87% da zona no instante da mudança (42,6 s) e volta ao nível anterior em poucos segundos; o total passa de 10 para 12 (+20%), mas o ganho foi aplicado no meio da passagem dos animais, então os dois itens a mais não podem ser atribuídos só à mudança de brilho. Nas maçãs, a mudança (aos 12,0 s) eleva o foreground de cerca de 40% para 55–70%, e ele não volta ao nível original nos 12 s restantes; o total cai de 59 para 36 (−39%). Uma explicação provável, não verificada, é que a zona fica quase toda marcada e os contornos se fundem em um só, o que reduz o número de eventos.